# Análise final: respostas às perguntas do objetivo

Este notebook responde às perguntas definidas no `00_Pre_Projeto`, consultando apenas a camada ouro. 

**Convenções usadas em todas as consultas:**
- Perguntas sobre compra e venda consideram apenas pedidos com status `delivered`, que são as compras efetivadas.
- Perguntas sobre avaliação consideram todos os pedidos avaliados, entregues ou não, para não excluir as avaliações de quem não recebeu o produto.
- Percentuais são calculados sobre o total da própria consulta.


## 1. Qual estado tem o maior número de compras online?

In [0]:
SELECT
  estado_cliente,
  COUNT(*)                                            AS compras,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS pct
FROM pospucrio.gold.pedidos
WHERE status = 'delivered'
GROUP BY estado_cliente
ORDER BY compras DESC

São Paulo concentra 42% das compras entregues no período (40.406 pedidos), mais que o triplo do Rio de Janeiro (12,8%), que aparece praticamente empatado com Minas Gerais (11,8%). Os três estados somam 67% das compras. Por região, o Sudeste reúne cerca de 69%, o Sul 14%, o Nordeste 9%, o Centro-Oeste 6% e o Norte menos de 2%.

Discussão. A explicação mais imediata seria o tamanho da população, já que os estados que mais compram estão entre os mais populosos. A comparação com a distribuição da população pelo Censo 2022 do IBGE mostra que ela explica apenas parte do resultado:

| Estado | % da população | % das compras |
| :--- | :---: | :---: |
| SP | 22 | 42 |
| MG | 10 | 12 |
| RJ | 8 | 13 |
| BA | 7 | 3 |
| RS | 5 | 6 |

Minas Gerais e Rio Grande do Sul compram em proporção próxima à de sua população. São Paulo compra quase o dobro de sua participação populacional, e o Rio de Janeiro também fica acima. A Bahia, quarto estado mais populoso, responde por cerca de metade das compras que seu tamanho sugeriria. A população, sozinha, não explica a distribuição.

Dois fatores, não medidos diretamente neste trabalho, são compatíveis com o resultado. O primeiro é a renda: a compra online depende de poder de consumo e de acesso a meios como o cartão de crédito, e São Paulo e Rio de Janeiro estão entre os estados de maior renda do país. O segundo é a proximidade dos vendedores: como mostra a pergunta 3, a maior parte dos vendedores da Olist está no Sudeste, sobretudo em São Paulo. Para quem compra perto de onde os produtos saem, isso tende a significar frete menor e entrega mais rápida. Os dados indicam que população, renda e proximidade atuam em conjunto, e que a concentração em São Paulo vai além do seu tamanho.

In [0]:
SELECT
  estado_cliente,
  COUNT(*)                                  AS pedidos,
  ROUND(AVG(frete_total), 2)                AS frete_medio,
  percentile_approx(dias_entrega, 0.5)      AS dias_entrega_mediana
FROM pospucrio.gold.pedidos
WHERE status = 'delivered'
GROUP BY estado_cliente
ORDER BY frete_medio ASC

A hipótese da proximidade é confirmada pelos dados de frete e prazo. Compradores de São Paulo pagam frete médio de R$ 17,33 e recebem seus pedidos em 7 dias (mediana). No restante do Sudeste e do Sul, o frete fica entre R$ 23 e R$ 25, com prazos de 10 a 14 dias; no Nordeste e no Norte, entre R$ 30 e R$ 49, com 16 a 26 dias. O caso do Rio de Janeiro indica que a proximidade não é o único fator: com frete praticamente igual ao de Minas Gerais, o estado compra bem acima de sua participação populacional, enquanto Minas compra na proporção da sua. Como estados mais próximos de São Paulo tendem também a ter renda mais alta, os dados não permitem separar o efeito do frete do efeito da renda.

## 2. Qual estado tem o maior gasto com compras online?


In [0]:
SELECT
  estado_cliente,
  SUM(valor_pago)                                          AS gasto_total,
  ROUND(100 * SUM(valor_pago) / SUM(SUM(valor_pago)) OVER (), 2) AS pct,
  ROUND(AVG(valor_pago), 2)                                AS ticket_medio,
  percentile_approx(valor_pago, 0.5)                       AS ticket_mediano
FROM pospucrio.gold.pedidos
WHERE status = 'delivered'
GROUP BY estado_cliente
ORDER BY gasto_total DESC

São Paulo tem o maior gasto total, com R$ 5,76 milhões (37,4% do valor pago em compras entregues), seguido de Rio de Janeiro (R$ 2,05 milhões, 13,3%) e Minas Gerais (R$ 1,81 milhão, 11,8%). A ordem acompanha a do número de compras, mas São Paulo perde participação: responde por 42% das compras e 37,4% do gasto. Seu ticket é o menor do país, com média de R$ 142,47 e mediana de R$ 93,64.

O valor pago inclui o frete, que varia muito entre estados.

In [0]:
SELECT
  estado_cliente,
  ROUND(AVG(valor_pago), 2)                          AS ticket_medio,
  ROUND(AVG(frete_total), 2)                         AS frete_medio,
  ROUND(AVG(valor_pago - frete_total), 2)            AS ticket_sem_frete,
  ROUND(100 * AVG(frete_total / valor_pago), 1)      AS pct_frete_no_ticket
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND valor_pago > 0
GROUP BY estado_cliente
ORDER BY ticket_sem_frete DESC

 Separando as duas parcelas, o padrão permanece: mesmo sem o frete, o valor médio das compras cresce com a distância de São Paulo. Vai de R$ 125 em São Paulo e cerca de R$ 135 a R$ 145 no restante do Sudeste, no Sul e no Distrito Federal, até valores entre R$ 150 e R$ 218 no Nordeste e no Norte.

A explicação mais consistente com os dados é que o frete muda o que vale a pena comprar online. Quando o frete é caro e a entrega demora, a compra pela internet só compensa para produtos de valor mais alto, nos quais o custo do transporte pesa proporcionalmente menos. Onde o frete é barato e a entrega rápida, como em São Paulo, compras de menor valor também se tornam viáveis. Ainda assim, a diluição é parcial: o frete representa 18% do valor pago em São Paulo, entre 21% e 23% no restante do Sudeste e no Sul, e entre 24% e 32% no Norte e no Nordeste. Os estados com menos de 100 pedidos (Roraima, Amapá e Acre) não foram considerados individualmente.

## 3. Qual estado mais vende em número de vendas?

In [0]:
SELECT
  estado_vendedor,
  COUNT(*)                                            AS itens_vendidos,
  COUNT(DISTINCT order_id)                            AS pedidos_atendidos,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS pct_itens
FROM pospucrio.gold.itens
WHERE status = 'delivered'
GROUP BY estado_vendedor
ORDER BY itens_vendidos DESC

São Paulo lidera com folga: 78.419 itens vendidos, 71,4% do total, em 68.489 pedidos. Minas Gerais (7,8%) e Paraná (7,7%) aparecem praticamente empatados em segundo, seguidos por Rio de Janeiro (4,2%), Santa Catarina (3,6%) e Rio Grande do Sul (2,0%). Sul e Sudeste concentram 97% dos itens vendidos. O Nordeste inteiro responde por pouco mais de 1%, e o Norte, por menos de 0,1%.

Discussão. A concentração é muito maior do lado de quem vende do que do lado de quem compra. Isso se explica pelo modelo de negócio da empresa no período: a Olist conectava pequenos e médios lojistas aos grandes marketplaces. Os vendedores são, portanto, os clientes da Olist, e a distribuição por estado mostra onde a empresa conquistou sua base de lojistas, e não onde está a produção ou o comércio do país.

Nessa leitura, a liderança de São Paulo reflete o tamanho do maior mercado de lojistas do Brasil. A força do Paraná, bem acima do seu peso populacional, é compatível com a origem da empresa, sediada em Curitiba.

A comparação com a pergunta 1 mostra o saldo de cada estado. São Paulo responde por 42% das compras e 71% das vendas, e o Paraná, por 5% e 8%: os dois abastecem o restante do país. O Rio de Janeiro faz o caminho inverso, com 13% das compras e 4% das vendas, e o Nordeste compra cerca de 9% e vende pouco mais de 1%. Essa concentração dos vendedores completa a explicação das perguntas anteriores: quem compra longe do Sudeste e do Sul recebe produtos que viajam grandes distâncias, o que se reflete em fretes maiores e prazos mais longos.

Uma questão em aberto: por que Minas Gerais vende tanto mais que o Rio de Janeiro? A proximidade explica a liderança de São Paulo, mas não a ordem seguinte. Minas Gerais e Rio de Janeiro têm frete praticamente igual para seus compradores e são vizinhos de São Paulo. Minas tem cerca de 28% mais habitantes que o Rio, mas vende 85% mais itens. O Rio, segundo maior comprador do país, é apenas o quarto vendedor. O tamanho do estado não basta para explicar a diferença.

Uma hipótese é que as bases de lojistas de cada estado estivessem em momentos diferentes. Para testá-la, a consulta a seguir mostra a participação de cada estado nas vendas ao longo dos trimestres.

In [0]:
SELECT
  concat(ano, '-T', trimestre)                                     AS periodo,
  COUNT(*)                                                         AS itens,
  ROUND(100 * COUNT_IF(estado_vendedor = 'SP') / COUNT(*), 1)      AS pct_sp,
  ROUND(100 * COUNT_IF(estado_vendedor = 'MG') / COUNT(*), 1)      AS pct_mg,
  ROUND(100 * COUNT_IF(estado_vendedor = 'PR') / COUNT(*), 1)      AS pct_pr,
  ROUND(100 * COUNT_IF(estado_vendedor = 'RJ') / COUNT(*), 1)      AS pct_rj
FROM pospucrio.gold.itens
WHERE status = 'delivered' AND data_compra < '2018-07-01'
GROUP BY ano, trimestre
ORDER BY ano, trimestre

À primeira vista, Minas Gerais perde espaço: sua participação cai de 12,1% no primeiro trimestre de 2017 para 6,7% no segundo trimestre de 2018. Participação, porém, não é volume. No mesmo período, o total de itens vendidos por trimestre quadruplicou, e uma fatia menor de um total maior pode representar mais vendas. A consulta seguinte mostra os volumes absolutos.

In [0]:
SELECT
  concat(ano, '-T', trimestre)          AS periodo,
  COUNT(*)                              AS total,
  COUNT_IF(estado_vendedor = 'SP')      AS sp,
  COUNT_IF(estado_vendedor = 'MG')      AS mg,
  COUNT_IF(estado_vendedor = 'PR')      AS pr,
  COUNT_IF(estado_vendedor = 'RJ')      AS rj
FROM pospucrio.gold.itens
WHERE status = 'delivered' AND data_compra < '2018-07-01'
GROUP BY ano, trimestre
ORDER BY ano, trimestre

Os volumes mudam a leitura. Minas Gerais não perdeu vendas: mais que dobrou ao longo de 2017, de cerca de 690 para 1.590 itens por trimestre, e a partir do fim do ano se estabilizou em torno de 1.500. A queda na participação se deve ao crescimento mais rápido dos outros estados, principalmente São Paulo, cuja fatia subiu de 64% para 73%. O mesmo vale para a aparente queda de todos os estados no segundo e terceiro trimestres de 2017: nenhum deles vendeu menos, São Paulo é que cresceu mais que a média.

O Rio de Janeiro faz o percurso oposto ao de Minas. Começa o período com uma base pequena, cerca de 230 itens por trimestre, e é o estado que mais cresce, chegando a cerca de 1.110 no segundo trimestre de 2018, quase cinco vezes mais, com aceleração a partir do fim de 2017.

A diferença entre os dois estados vem, em boa parte, de uma vantagem de largada. Quando a série começa, Minas já era o segundo polo vendedor da Olist, com base estabelecida; o Rio estava em expansão e ainda não havia alcançado. Ao fim do período, a distância entre os dois diminuía.

O Paraná apresenta um comportamento isolado: um pico no primeiro trimestre de 2018, quando passa de cerca de 1.410 para 2.360 itens, voltando a 1.610 no trimestre seguinte. Um salto e um recuo dessa dimensão em um único trimestre sugerem um evento pontual, como a atuação concentrada de um grande vendedor, e não uma tendência.

Os dados não permitem explicar por que Minas Gerais começou o período com uma base de lojistas maior, nem confirmar a causa do pico do Paraná. As duas questões dependeriam de informações sobre os vendedores individuais, que o modelo atual não inclui.

## 4. Qual estado mais vende em valor?

In [0]:
SELECT
  estado_vendedor,
  SUM(preco)                                               AS valor_vendido,
  ROUND(100 * SUM(preco) / SUM(SUM(preco)) OVER (), 2)     AS pct,
  ROUND(AVG(preco), 2)                                     AS preco_medio
FROM pospucrio.gold.itens
WHERE status = 'delivered'
GROUP BY estado_vendedor
ORDER BY valor_vendido DESC

São Paulo lidera também em valor, com R$ 8,49 milhões, 64,4% do total vendido. Em seguida vêm Paraná (R$ 1,23 milhão, 9,3%), Minas Gerais (R$ 976 mil, 7,4%) e Rio de Janeiro (R$ 813 mil, 6,2%). A ordem acompanha a do número de itens vendidos, com uma inversão: o Paraná passa Minas Gerais.

A diferença está no preço médio dos itens. Paraná e Minas vendem quase a mesma quantidade (8.445 e 8.579 itens), mas o Paraná vende itens de R$ 145, em média, contra R$ 114 de Minas. O Rio de Janeiro segue a mesma lógica: responde por 4,2% dos itens e 6,2% do valor, com o maior preço médio entre os estados de maior volume, R$ 175.

São Paulo, o estado que mais vende, tem um dos menores preços médios, R$ 108. O resultado conversa com a pergunta 2, em que os compradores paulistas fazem as compras de menor valor do país. Uma hipótese compatível com as duas pontas é que produtos baratos circulem principalmente perto de onde são vendidos, onde o frete baixo torna a compra viável, e que os produtos que percorrem longas distâncias sejam os de maior valor.

In [0]:
SELECT
  categoria,
  COUNT(*)                   AS itens,
  ROUND(AVG(preco), 2)       AS preco_medio,
  SUM(preco)                 AS valor_vendido
FROM pospucrio.gold.itens
WHERE status = 'delivered' AND estado_vendedor = 'BA'
GROUP BY categoria
ORDER BY valor_vendido DESC
LIMIT 10

A Bahia chama atenção pelo preço médio de R$ 445,85, o maior entre os estados com volume relevante. A explicação é a composição do que é vendido: com apenas 19 vendedores no período, os itens baianos se concentram em eletrônicos. Computadores respondem sozinhos por 60% do valor vendido pelo estado, com preço médio de R$ 1.112, e, somados a telefonia e informática, chegam a 91%. Estados com menos de 100 itens vendidos, como Paraíba e Rondônia, também apresentam médias altas, mas amostras tão pequenas não sustentam comparação e não foram consideradas

## 5. Quais são as notas dadas pelos clientes?

In [0]:
SELECT
  nota,
  COUNT(*)                                            AS avaliacoes,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)    AS pct
FROM pospucrio.gold.pedidos
WHERE nota IS NOT NULL
GROUP BY nota
ORDER BY nota

In [0]:
SELECT
  estado_cliente,
  COUNT(*)                                                AS avaliacoes,
  ROUND(AVG(nota), 2)                                     AS nota_media,
  ROUND(100 * COUNT_IF(nota <= 2) / COUNT(*), 1)          AS pct_notas_1_e_2
FROM pospucrio.gold.pedidos
WHERE nota IS NOT NULL
GROUP BY estado_cliente
ORDER BY nota_media DESC

Foram avaliados 98.349 pedidos, com média geral de 4,09. A distribuição é polarizada: 57,8% dos pedidos receberam nota 5 e 19,3% nota 4, enquanto a nota 1 (11,5%) é mais frequente que as notas 2 (3,2%) e 3 (8,2%) somadas.

Por estado, os de maior volume ficam próximos entre si: São Paulo e Paraná com 4,18, Minas Gerais e Rio Grande do Sul com 4,14. As menores médias estão no Rio de Janeiro e em estados do Nordeste e do Norte. Amazonas, Amapá e Roraima, com menos de 150 avaliações cada, apresentam médias instáveis e não foram considerados.

Duas questões surgem desses resultados. A primeira é a polarização. Ela poderia ser atribuída à tendência de avaliar apenas quando se está muito satisfeito ou muito insatisfeito, mas essa explicação não se aplica aqui: 99% dos pedidos têm avaliação. Se quase todos os clientes avaliaram, a concentração nos extremos deve vir da própria experiência de compra.

A segunda é a posição do Rio de Janeiro. Os estados com as piores notas são, em sua maioria, do Nordeste e do Norte, onde os prazos de entrega são mais longos (pergunta 1). O Rio de Janeiro, com prazo mediano de 12 dias, próximo ao de Minas Gerais, tem 20,6% de notas 1 e 2, contra 13,2% de

In [0]:
SELECT
  CASE WHEN dias_entrega <= 7  THEN 'até 7 dias'
       WHEN dias_entrega <= 14 THEN '8 a 14 dias'
       WHEN dias_entrega <= 21 THEN '15 a 21 dias'
       WHEN dias_entrega <= 30 THEN '22 a 30 dias'
       ELSE 'mais de 30 dias' END                   AS prazo,
  COUNT(*)                                          AS avaliacoes,
  ROUND(AVG(nota), 2)                               AS nota_media,
  ROUND(100 * COUNT_IF(nota <= 2) / COUNT(*), 1)    AS pct_notas_1_e_2
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND nota IS NOT NULL AND dias_entrega IS NOT NULL
GROUP BY 1
ORDER BY MIN(dias_entrega)

In [0]:
SELECT
  estado_cliente,
  COUNT(*)                                                         AS pedidos,
  ROUND(100 * COUNT_IF(status <> 'delivered') / COUNT(*), 1)       AS pct_nao_entregue,
  percentile_approx(dias_entrega, 0.5)                             AS dias_entrega_mediana,
  ROUND(AVG(CASE WHEN status = 'delivered' THEN nota END), 2)      AS nota_entregues,
  ROUND(AVG(CASE WHEN status <> 'delivered' THEN nota END), 2)     AS nota_nao_entregues
FROM pospucrio.gold.pedidos
WHERE estado_cliente IN ('SP', 'MG', 'RJ', 'PR', 'RS', 'BA')
GROUP BY estado_cliente
ORDER BY nota_entregues DESC

A nota cai de forma contínua com o prazo de entrega e despenca depois de três semanas. Acima de 30 dias, quase dois em cada três pedidos recebem nota 1 ou 2.

A polarização se explica pela experiência: a maior parte das compras chega em prazo razoável e recebe nota alta; as que atrasam muito recebem nota baixa. As notas mais baixas do Nordeste acompanham os prazos mais longos da região, e se ligam à concentração dos vendedores no Sudeste e no Sul observada na pergunta 3.

O prazo mediano, porém, não explica o Rio de Janeiro. A consulta a seguir compara os estados de maior volume em duas dimensões: a proporção de pedidos não entregues e a nota separada entre pedidos entregues e não entregues.

Pedidos não entregues recebem nota média entre 1,64 e 1,86 em todos os estados: a frustração de não receber é a mesma em qualquer lugar. A Bahia se explica pelo prazo, com a mediana mais longa da tabela e nota compatível com essa faixa.

O Rio de Janeiro é explicado apenas em parte. Tem a maior taxa de pedidos não entregues entre os estados de maior volume, 3,8%, o que puxa sua média para baixo. Mas, mesmo considerando apenas os pedidos entregues, sua nota (3,97) fica abaixo da de Minas Gerais e do Rio Grande do Sul (4,19), que entregam em prazo mediano semelhante ou maior. Como a mediana esconde os casos extremos, a consulta a seguir mede a proporção de entregas acima de 21 dias, faixa em que a nota despenca.

In [0]:
SELECT
  estado_cliente,
  ROUND(100 * COUNT_IF(dias_entrega > 21) / COUNT(*), 1) AS pct_acima_21_dias
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND dias_entrega IS NOT NULL
  AND estado_cliente IN ('SP', 'MG', 'RJ', 'PR', 'RS', 'BA')
GROUP BY estado_cliente
ORDER BY pct_acima_21_dias DESC

O Rio de Janeiro tem mais que o dobro da proporção de entregas longas de Minas Gerais (18,5% contra 8,0%), o que explica boa parte da diferença de nota entre os dois estados. A comparação com o Rio Grande do Sul, porém, mostra que o prazo não explica tudo: com proporção semelhante de entregas longas (16,8%), os compradores gaúchos avaliam melhor os pedidos entregues (4,19 contra 3,97).

Uma hipótese que concilia os dois casos é a expectativa de prazo. Para um estado distante do principal polo vendedor, como o Rio Grande do Sul, o prazo prometido na compra tende a ser mais longo, e uma entrega de três semanas pode estar dentro do combinado. No Rio de Janeiro, mais próximo de São Paulo, a mesma entrega tende a representar um atraso em relação ao prometido. A insatisfação estaria ligada ao atraso sobre a data estimada, e não ao prazo absoluto. O teste dessa hipótese exigiria incorporar a data estimada de entrega à camada ouro e fica registrado como trabalho futuro.

## 6. Qual o método de pagamento mais utilizado?


In [0]:
SELECT
  metodo_principal,
  COUNT(*)                                            AS pedidos,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS pct
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND metodo_principal IS NOT NULL
GROUP BY metodo_principal
ORDER BY pedidos DESC

O cartão de crédito é o método principal de 77,0% das compras entregues, seguido pelo boleto, com 19,9%. Voucher e cartão de débito aparecem com cerca de 1,5% cada.

Os dados são de 2016 a 2018, antes do lançamento do Pix, em novembro de 2020. As alternativas ao cartão de crédito para compras online eram, basicamente, o boleto e o cartão de débito. A distribuição descreve esse período; com o Pix disponível, é provável que ela fosse diferente.

O voucher aparece como método principal apenas nos pedidos pagos exclusivamente com ele. Quando combinado com outro meio, o método principal é o outro, conforme a regra definida na modelagem. Por isso, a presença do voucher nos pagamentos é maior do que 1,55%: ele também aparece em cerca de 2.200 pedidos pagos com cartão de crédito.

Uma explicação possível para a predominância do cartão de crédito é o parcelamento, que só ele permite. A consulta a seguir cruza o método com a forma de pagamento para testar essa hipótese.

In [0]:
SELECT
  metodo_principal,
  forma_pagamento,
  COUNT(*)                                                                       AS pedidos,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY metodo_principal), 1)  AS pct_do_metodo
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND forma_pagamento IS NOT NULL
GROUP BY metodo_principal, forma_pagamento
ORDER BY metodo_principal, forma_pagamento

O parcelamento explica parte da predominância do cartão: 66,8% das compras no cartão de crédito foram parceladas, e todas as compras parceladas do período foram feitas nele. Boleto, débito e voucher são sempre pagos em parcela única.

Mas o parcelamento não explica tudo. Mesmo entre as 46.709 compras pagas à vista, o cartão de crédito é o meio mais usado, com 52,7%, contra 41,0% do boleto, 3,2% do débito e 3,2% do voucher. Os compradores preferiam o cartão de crédito mesmo quando não parcelavam. Os dados não informam o motivo dessa preferência.

## 7. Qual categoria apresenta mais demanda?

In [0]:
SELECT
  categoria,
  COUNT(*)                                            AS itens_vendidos,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS pct_itens,
  SUM(preco)                                          AS valor_vendido,
  ROUND(AVG(preco), 2)                                AS preco_medio
FROM pospucrio.gold.itens
WHERE status = 'delivered'
GROUP BY categoria
ORDER BY itens_vendidos DESC
LIMIT 15

A demanda é dispersa. Nenhuma categoria alcança 10% dos itens vendidos, e as cinco primeiras somam 40,5%. A liderança de cama, mesa e banho existe, mas com margem pequena sobre as seguintes.

Volume e valor não coincidem. Beleza e saúde vende menos itens que cama, mesa e banho, mas gera mais receita, com preço médio de R$ 130,50 contra R$ 93,46. Relógios e presentes, sétima categoria em volume, é a segunda em valor entre as quinze analisadas, com o maior preço médio do grupo, R$ 198,71. No outro extremo, eletrônicos e telefonia vendem volumes relevantes de itens baratos, com preços médios de R$ 56,79 e R$ 70,02.

O preço médio indica onde se concentra o valor das vendas, mas não a rentabilidade para o vendedor, que depende de margens e custos não presentes nos dados.

## 8. Houve mudança na média das avaliações ao longo dos trimestres?

In [0]:
SELECT
  concat(ano, '-T', trimestre)                            AS periodo,
  COUNT(*)                                                AS avaliacoes,
  ROUND(AVG(nota), 2)                                     AS nota_media,
  ROUND(100 * COUNT_IF(nota <= 2) / COUNT(*), 1)          AS pct_notas_1_e_2
FROM pospucrio.gold.pedidos
WHERE nota IS NOT NULL
  AND data_compra < '2018-07-01'
GROUP BY ano, trimestre
ORDER BY ano, trimestre

A nota melhora ao longo de 2017 até o terceiro trimestre, cai nos dois trimestres seguintes e se recupera no segundo trimestre de 2018. A variação parece pequena na média, de 4,20 para 3,87, mas é expressiva na proporção de notas 1 e 2, que passa de 12,0% para 19,7%: quase o dobro de clientes muito insatisfeitos.

A queda coincide com o período de maior crescimento de pedidos. O quarto trimestre de 2017 inclui a Black Friday, em novembro, mês com o maior salto de volume da série, e o primeiro trimestre de 2018 é o de maior número de pedidos do período. Como a pergunta 5 mostrou que a nota depende fortemente do prazo de entrega, a hipótese é que o crescimento do volume tenha superado a capacidade de entrega. A consulta a seguir verifica a evolução dos prazos e das entregas não realizadas no mesmo período.

In [0]:
SELECT
  concat(ano, '-T', trimestre)                                                        AS periodo,
  COUNT(*)                                                                            AS pedidos,
  ROUND(100 * COUNT_IF(status <> 'delivered') / COUNT(*), 1)                          AS pct_nao_entregue,
  percentile_approx(CASE WHEN status = 'delivered' THEN dias_entrega END, 0.5)        AS dias_entrega_mediana,
  ROUND(100 * COUNT_IF(status = 'delivered' AND dias_entrega > 21)
            / COUNT_IF(status = 'delivered' AND dias_entrega IS NOT NULL), 1)         AS pct_acima_21_dias
FROM pospucrio.gold.pedidos
WHERE data_compra < '2018-07-01'
GROUP BY ano, trimestre
ORDER BY ano, trimestre

Os prazos confirmam a hipótese. A proporção de entregas acima de 21 dias, faixa em que a nota despenca (pergunta 5), sobe de 7,4% no terceiro trimestre de 2017 para 16,6% no quarto e 21,9% no primeiro de 2018, e volta a 8,0% no segundo. O prazo mediano segue o mesmo movimento, de 10 para 13 dias e depois 9. A proporção de notas 1 e 2 acompanha esse desenho nos mesmos trimestres.

A taxa de pedidos não entregues, ao contrário, cai durante todo o período, de 5,9% para 1,7%. A piora das notas no fim de 2017 e início de 2018 não se deve, portanto, a pedidos que não chegaram, e sim a pedidos que chegaram tarde. Já no início da série a situação é inversa: com prazos dentro do normal, a nota mais baixa do primeiro trimestre de 2017 acompanha a maior taxa de pedidos não entregues do período.

A leitura que emerge é a de uma operação em crescimento. A Olist melhorou continuamente a taxa de entrega, mas o pico de volume do fim de 2017, com a Black Friday, e do início de 2018 pressionou os prazos. Quando os prazos se normalizaram, no segundo trimestre de 2018, a nota voltou ao patamar mais alto da série. A série tem apenas seis pontos, o que não permite estabelecer causalidade por si só; mas o resultado é consistente com a relação entre prazo e nota observada, em escala muito maior, na pergunta 5.

## 9. Qual o crescimento entre o primeiro semestre de 2017 e o de 2018?

In [0]:
WITH semestres AS (
  SELECT
    ano,
    COUNT(*)          AS pedidos,
    SUM(valor_pago)   AS valor_pago
  FROM pospucrio.gold.pedidos
  WHERE status = 'delivered' AND semestre = 1
  GROUP BY ano
)
SELECT
  s17.pedidos                                                        AS pedidos_2017,
  s18.pedidos                                                        AS pedidos_2018,
  ROUND(100 * (s18.pedidos - s17.pedidos) / s17.pedidos, 1)          AS crescimento_pedidos_pct,
  s17.valor_pago                                                     AS valor_2017,
  s18.valor_pago                                                     AS valor_2018,
  ROUND(100 * (s18.valor_pago - s17.valor_pago) / s17.valor_pago, 1) AS crescimento_valor_pct
FROM semestres s17
CROSS JOIN semestres s18
WHERE s17.ano = 2017 AND s18.ano = 2018

Entre o primeiro semestre de 2017 e o primeiro semestre de 2018, o número de pedidos entregues cresceu 189%, de 13.933 para 40.273, e o valor pago cresceu 184,8%, de R$ 2,26 milhões para R$ 6,44 milhões.

| Indicador | 1º sem. 2017 | 1º sem. 2018 | Crescimento |
|---|---|---|---|
| Pedidos entregues | 13.933 | 40.273 | 189,0% |
| Valor pago (R$) | 2.261.458,22 | 6.439.657,06 | 184,8% |
| Ticket médio (R$) | 162,31 | 159,90 | -1,5% |

A Olist quase triplicou de tamanho em um ano. Como o valor cresceu um pouco menos que o número de pedidos, o ticket médio recuou levemente, de R$ 162,31 para R$ 159,90: o crescimento veio do aumento no número de compras, e não de compras de maior valor.

A magnitude deve ser lida à luz da base de comparação. Os dados começam em setembro de 2016, e o primeiro semestre de 2017 corresponde a uma fase inicial da operação. Taxas dessa ordem são típicas de uma empresa em expansão a partir de uma base pequena, e não permitem conclusões sobre o comércio eletrônico brasileiro como um todo.

O crescimento ajuda a explicar a queda de satisfação observada na pergunta 8. O aumento de volume no fim de 2017 e no início de 2018 coincide com o alongamento dos prazos de entrega e a piora das notas, que só se recuperam no segundo trimestre de 2018. O crescimento acelerado teve, assim, um custo operacional temporário, absorvido ao longo de alguns meses.

## 10. Qual a proporção de compras pagas à vista e de forma parcelada?

In [0]:
SELECT
  forma_pagamento,
  COUNT(*)                                            AS pedidos,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS pct
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND forma_pagamento IS NOT NULL
GROUP BY forma_pagamento

As compras entregues se dividem de forma quase igual: 51,45% parceladas (49.500 pedidos) e 48,55% à vista (46.709).

| Forma de pagamento | Pedidos | % |
|---|---|---|
| Parcelado | 49.500 | 51,45 |
| À vista | 46.709 | 48,55 |

A proporção próxima de metade subestima a preferência pelo parcelamento. Como mostrou a pergunta 6, apenas o cartão de crédito permite parcelar: boleto, débito e voucher são sempre pagos em parcela única. As 22.116 compras feitas por esses meios entram na conta como à vista por uma limitação do meio de pagamento, e não por escolha do comprador.

Considerando apenas quem podia escolher, a preferência fica clara: 66,8% das compras no cartão de crédito foram parceladas. Entre os compradores que tinham a opção, dois em cada três dividiram o pagamento.

## 11. Qual o valor médio das compras à vista e das parceladas?

In [0]:
SELECT
  forma_pagamento,
  COUNT(*)                                  AS pedidos,
  ROUND(AVG(valor_pago), 2)                 AS valor_medio,
  percentile_approx(valor_pago, 0.5)        AS valor_mediano
FROM pospucrio.gold.pedidos
WHERE status = 'delivered' AND forma_pagamento IS NOT NULL
GROUP BY forma_pagamento

As compras parceladas têm valor médio de R$ 197,19 e mediana de R$ 134,49; as compras à vista, R$ 120,21 e R$ 79,24. A compra parcelada custa, em média, 64% mais, e a diferença entre as medianas é de 70%.

| Forma de pagamento | Pedidos | Valor médio (R$) | Valor mediano (R$) |
|---|---|---|---|
| À vista | 46.709 | 120,21 | 79,24 |
| Parcelado | 49.500 | 197,19 | 134,49 |

O resultado confirma a expectativa de que compras de maior valor são parceladas com mais frequência. Os juros de parcelamento, incluídos no valor pago, não explicam a diferença: a análise de qualidade mostrou que apenas cerca de 250 pedidos pagaram mais que o valor dos itens, com diferença média de R$ 12. Na maior parte dos casos, o parcelamento era sem juros, e a diferença reflete o valor dos produtos.

As médias e medianas mostram que as compras parceladas são mais caras, mas não indicam a partir de qual valor o parcelamento passa a predominar, porque os dois grupos se sobrepõem. A consulta a seguir mede a proporção de compras parceladas por faixa de valor, considerando apenas as compras no cartão de crédito, único meio que permite parcelar.

In [0]:
SELECT
  CASE WHEN valor_pago < 50  THEN 'até R$ 50'
       WHEN valor_pago < 100 THEN 'R$ 50 a 100'
       WHEN valor_pago < 150 THEN 'R$ 100 a 150'
       WHEN valor_pago < 200 THEN 'R$ 150 a 200'
       WHEN valor_pago < 300 THEN 'R$ 200 a 300'
       WHEN valor_pago < 500 THEN 'R$ 300 a 500'
       ELSE 'acima de R$ 500' END                                        AS faixa_valor,
  COUNT(*)                                                               AS pedidos,
  ROUND(100 * COUNT_IF(forma_pagamento = 'parcelado') / COUNT(*), 1)     AS pct_parcelado
FROM pospucrio.gold.pedidos
WHERE status = 'delivered'
  AND metodo_principal = 'cartão de crédito'
  AND forma_pagamento IS NOT NULL
GROUP BY 1
ORDER BY MIN(valor_pago)

A proporção de compras parceladas cresce de forma contínua com o valor, sem um ponto de corte nítido. O parcelamento passa a ser maioria já na faixa de R$ 50 a 100, com 56%, e ultrapassa 80% a partir de R$ 150.

O dado mais expressivo está na faixa mais baixa: 38,7% das compras de até R$ 50 no cartão de crédito foram parceladas. Se o parcelamento fosse usado apenas para viabilizar compras caras, essa proporção seria próxima de zero. O resultado indica que, para esses compradores, parcelar era um hábito, adotado mesmo em compras pequenas e cada vez mais frequente à medida que o valor aumenta.

## Análise complementar: avaliação de pedidos entregues e não entregues

Não responde a uma pergunta do objetivo, mas testa uma decisão do modelo: manter os pedidos não entregues na tabela para as perguntas de avaliação. Se a nota média dos não entregues for muito menor, excluí-los teria elevado artificialmente a satisfação medida.

In [0]:
SELECT
  CASE WHEN status = 'delivered' THEN 'entregue' ELSE 'não entregue' END AS situacao,
  COUNT(*)                                                AS avaliacoes,
  ROUND(AVG(nota), 2)                                     AS nota_media,
  ROUND(100 * COUNT_IF(nota <= 2) / COUNT(*), 1)          AS pct_notas_1_e_2
FROM pospucrio.gold.pedidos
WHERE nota IS NOT NULL
GROUP BY 1

Pedidos entregues têm nota média de 4,16, com 12,8% de notas 1 e 2. Pedidos não entregues têm nota média de 1,75, com 78,0% de notas 1 e 2.

| Situação | Avaliações | Nota média | % notas 1 e 2 |
|---|---|---|---|
| Entregue | 95.568 | 4,16 | 12,8 |
| Não entregue | 2.781 | 1,75 | 78,0 |

A diferença é esperada, e o objetivo desta análise é medir o efeito de uma decisão da modelagem: manter os pedidos não entregues na tabela para as perguntas de avaliação, aplicando o filtro de entregues apenas nas perguntas de compra.

Os pedidos não entregues representam 2,8% dos pedidos avaliados, mas respondem por cerca de 15% de todas as notas 1 e 2. Se tivessem sido excluídos, a nota média geral subiria de 4,09 para 4,16, e a proporção de notas baixas cairia de 14,6% para 12,8%. Um filtro aparentemente neutro teria removido um sexto da insatisfação registrada, justamente a parcela associada à pior experiência de compra. O resultado confirma que a decisão de filtrar na consulta, e não na tabela, era necessária para uma medição correta da satisfação.

## Discussão geral




O objetivo do projeto era entender os hábitos de consumo e de venda na Olist entre 2017 e 2018. As respostas às perguntas, lidas em conjunto, descrevem um marketplace organizado em torno de um centro geográfico, e mostram como a distância desse centro molda o comportamento de quem compra e a satisfação com a compra.

**Um marketplace concentrado no Sudeste e no Sul.** Do lado de quem vende, a concentração é extrema: 71% dos itens saem de São Paulo, e Sul e Sudeste somam 97%. Como os vendedores são os clientes da Olist, essa distribuição mostra onde a empresa construiu sua base de lojistas. São Paulo, o maior mercado de lojistas do país, lidera. O Paraná, sede da empresa, aparece bem acima de seu peso populacional. Minas Gerais já era o segundo polo vendedor no início da série e se estabilizou, enquanto o Rio de Janeiro, com base menor, era o estado que mais crescia. Do lado de quem compra, a concentração é menor, mas o Sudeste ainda reúne 69% das compras, e São Paulo compra quase o dobro do que sua população sugeriria.

**A distância define como se compra.** Com os vendedores concentrados, o comprador distante recebe produtos que viajam mais. O frete médio vai de R$ 17 em São Paulo a quase R$ 49 na Paraíba, e o prazo mediano, de 7 para até 26 dias. Esse custo muda o comportamento de compra. Longe dos vendedores, compra-se menos e compra-se mais caro: o valor dos produtos, sem o frete, cresce com a distância, porque só compensa pagar um frete alto por itens de valor maior. Perto, compras pequenas se tornam viáveis, e São Paulo tem o menor ticket do país. A renda também contribui para as diferenças entre estados, mas, como estados mais próximos de São Paulo tendem a ter renda mais alta, os dados não permitem separar os dois efeitos.

**A satisfação é, sobretudo, uma avaliação da entrega.** A distribuição das notas é polarizada, e a explicação está no prazo: a nota média cai de 4,41 nas entregas de até 7 dias para 2,19 nas que passam de 30, e pedidos não entregues recebem, em média, 1,75. As notas mais baixas se concentram no Nordeste e no Norte, onde os prazos são mais longos. O mesmo mecanismo aparece no tempo: no fim de 2017 e início de 2018, o volume de pedidos atingiu seu pico, a proporção de entregas acima de 21 dias triplicou e a nota caiu para 3,87, recuperando-se quando os prazos se normalizaram. A empresa cresceu 189% em um ano, e a operação levou alguns meses para acompanhar. O Rio de Janeiro indica que o prazo absoluto não é tudo: com entregas longas em proporção semelhante à do Rio Grande do Sul, avalia pior. Uma hipótese é que a insatisfação dependa do atraso em relação ao prazo prometido, e não do prazo em si.

**O pagamento revela um hábito de parcelar.** O cartão de crédito responde por 77% das compras, e sua predominância se explica em parte pelo parcelamento, que só ele permite: 66,8% das compras no cartão foram parceladas. A proporção de parcelamento cresce com o valor, de 38,7% nas compras de até R$ 50 a 91,9% nas acima de R$ 500. Que quase quatro em cada dez compras pequenas sejam parceladas indica que parcelar era um hábito, e não apenas um recurso para compras caras. Ainda assim, mesmo entre as compras à vista, o cartão supera o boleto. Esses resultados descrevem o período anterior ao Pix, lançado em novembro de 2020, e a distribuição atual dos meios de pagamento provavelmente seria diferente.

**A demanda é dispersa.** Nenhuma categoria concentra mais de 10% dos itens vendidos. Cama, mesa e banho lidera em volume; beleza e saúde, em valor; relógios e presentes se destaca pelo maior preço médio entre as categorias de maior volume.

**Limites e questões em aberto.** Os resultados descrevem os clientes e vendedores de um único marketplace e não se generalizam para o comércio eletrônico brasileiro. Três questões ficaram sem resposta: por que Minas Gerais começou o período com uma base de lojistas maior que a do Rio de Janeiro; o que causou o pico isolado de vendas do Paraná no primeiro trimestre de 2018; e se a insatisfação dos compradores cariocas se deve ao atraso em relação ao prazo prometido. As duas primeiras exigiriam informações sobre os vendedores individuais; a terceira, a incorporação da data estimada de entrega à camada ouro.

## Autoavaliação

**Objetivo.** O objetivo era entender os hábitos de consumo e de venda na Olist entre 2017 e 2018, por meio de onze perguntas. Todas foram respondidas, e as respostas se conectam numa explicação coerente: a concentração dos vendedores no Sudeste e no Sul define o frete e o prazo de quem compra longe, o que por sua vez molda o que se compra e como se avalia a compra. Três questões levantadas durante a análise ficaram em aberto, por dependerem de dados que o modelo não inclui: a diferença entre as bases de lojistas de Minas Gerais e do Rio de Janeiro, o pico de vendas do Paraná no início de 2018 e a relação entre a insatisfação dos compradores cariocas e o prazo prometido.

**A formulação do problema.** A maior dificuldade do projeto veio antes de qualquer código: formular um problema. O trabalho pede que se crie um problema e depois se construa a solução para ele, sem que exista uma demanda real por trás. Para mim, é difícil partir de um problema inventado; costumo trabalhar a partir de problemas que já existem. Escolhi dados de comércio eletrônico por serem próximos da minha área de atuação, e as perguntas foram construídas a partir do que o dataset permitia responder. O resultado é um objetivo mais descritivo do que eu gostaria, sobre um período já distante: os dados vão até 2018, antes do Pix e de mudanças relevantes no comércio eletrônico. Considero essa a parte mais fraca do trabalho.

**Engenharia de dados.** O trabalho cumpriu seu papel principal, que era praticar o fluxo de um pipeline de ponta a ponta. A coleta foi feita com a biblioteca Python do Kaggle, com os arquivos preservados em um volume e as tabelas carregadas no bronze sem alteração. A arquitetura medalhão, que no início era um conceito, ficou clara na prática: o bronze preserva a origem, a prata garante que o dado é confiável e o ouro organiza o dado para as perguntas. Ficou claro também por que o diagnóstico de qualidade precisa vir antes da limpeza, e por que as regras de negócio devem ser decididas a partir do dado, e não presumidas.

A etapa mais difícil foi a modelagem da camada ouro. Precisei desenhar e redesenhar as tabelas até chegar a um modelo que fizesse sentido para as perguntas: entender que compra e venda têm granularidades diferentes, que juntá-las em uma tabela só duplicaria valores, e que cada regra de negócio precisava de uma decisão explícita. Mesmo depois de pronto, o modelo precisou ser revisto: durante a análise, uma hipótese sobre a influência da distância nas compras exigiu medir frete e prazo de entrega, que eu não havia previsto. Em vez de consultar a prata diretamente, estendi a camada ouro, mantendo-a como única fonte da análise. Outras decisões também mudaram ao longo do projeto: a regra de escolha da avaliação passou da data de criação para a data de resposta depois do diagnóstico, e a função de padronização de cidades foi ampliada quando os vendedores trouxeram padrões que não existiam nos clientes.

**Dificuldades.** Além da formulação do problema e da modelagem, tive dificuldade com recursos que não fazem parte do meu dia a dia: a diferença entre executar código no driver e no Spark, as restrições do ambiente serverless, a sintaxe de funções e expressões regulares em SQL e a orquestração do pipeline em jobs. Foram dificuldades de prática, que tendem a diminuir com o uso.

**O que poderia ter sido melhor.** As respostas são apresentadas principalmente em tabelas e texto. Gráficos mais bem escolhidos e um dashboard sobre a camada ouro comunicariam os resultados de forma mais clara, sobretudo as relações entre prazo, frete e nota, que são o centro da análise.

**Trabalhos futuros.**
- Apresentar as respostas com gráficos mais adequados e um dashboard sobre a camada ouro, em vez de tabelas e texto.
- Em um novo projeto, analisar dados recentes de comércio eletrônico e compará-los com os resultados deste trabalho, para identificar quais padrões se mantêm e quais mudaram, como o efeito do Pix sobre os meios de pagamento.
- Aprofundar o uso de Python na análise e visualização de dados, com as bibliotecas Pandas, Matplotlib e Seaborn. Neste projeto, a análise foi feita basicamente em SQL, e o Python ficou restrito à coleta dos dados.